In [6]:
import pandas as pd

pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

In [7]:
# Read the CSV file into a DataFrame
df = pd.read_csv("IJ-044.csv")

# Display the first 5 rows
print(df.head().to_markdown(index=False, numalign="left", stralign="left"))

# Print the column names and their data types
print(df.info())

| Data de Produção   | Cód. Ordem   | Cód. Recurso   | Cód. Produto   | Qtd. Produzida   | Qtd. Refugada   | Qtd. Retrabalhada   | Fator Un.   | Cód. Un.   | Descrição da massa (Composto)   | Consumo de massa no item em (Kg/100pçs)   |
|:-------------------|:-------------|:---------------|:---------------|:-----------------|:----------------|:--------------------|:------------|:-----------|:--------------------------------|:------------------------------------------|
| 2023-08-21         | 2680830      | IJ-044         | SA08395        | 239              | 4               | 0                   | 1           | PC         | N-142/67                        | 2.869                                     |
| 2023-08-21         | 2680830      | IJ-044         | SA08395        | 158              | 6               | 0                   | 1           | PC         | N-142/67                        | 2.869                                     |
| 2023-08-21         | 2680830      | IJ-044         | S

In [8]:
# Get all unique values from `Cód. Produto`
unique_values = df["Cód. Produto"].unique()

# Check the number of unique values in `Cód. Produto`
if len(unique_values) > 50:
    # If there are too many unique values, sample the top 50
    top_occurring_values = df["Cód. Produto"].value_counts().head(50).index.tolist()
    print(top_occurring_values)
else:
    # Otherwise print all unique valus in `Cód. Produto`
    print(unique_values)

['SA08395' 'SA08400' 'SA07321' 'SA01801' 'SA02941' '11974' 'SA04416'
 'SA11684' 'SA02938' 'SA00436' 'SA05032' 'SA18476' 'SA01974' 'SA02910'
 'SA03682' 'SA07337' 'SA04417' 'SA02583' 'SA02900' 'SA07339' 'SA02619'
 'SA07291' 'SA02973' 'SA15829' 'SA07371']


In [9]:
# coluna Cod Produto como variável alvo para a classificação e Qtd_Produzida como variável alvo para a regressão.

In [10]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import accuracy_score, r2_score

# Drop unnecessary columns
df = df.drop(
    [
        "Data de Produção",
        "Cód. Ordem",
        "Cód. Recurso",
        "Cód. Un.",
        "Descrição da massa (Composto)",
    ],
    axis=1,
)

# One-hot encode `Cód. Produto`
df = pd.get_dummies(df, columns=["Cód. Produto"])

# Split data into train and test sets
X = df.drop("Qtd. Produzida", axis=1)
y = df["Qtd. Produzida"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42
)

# Scale the data
scaler = MinMaxScaler()
X_train[
    [
        "Qtd. Refugada",
        "Qtd. Retrabalhada",
        "Fator Un.",
        "Consumo de massa no item em (Kg/100pçs)",
    ]
] = scaler.fit_transform(
    X_train[
        [
            "Qtd. Refugada",
            "Qtd. Retrabalhada",
            "Fator Un.",
            "Consumo de massa no item em (Kg/100pçs)",
        ]
    ]
)
X_test[
    [
        "Qtd. Refugada",
        "Qtd. Retrabalhada",
        "Fator Un.",
        "Consumo de massa no item em (Kg/100pçs)",
    ]
] = scaler.transform(
    X_test[
        [
            "Qtd. Refugada",
            "Qtd. Retrabalhada",
            "Fator Un.",
            "Consumo de massa no item em (Kg/100pçs)",
        ]
    ]
)

In [11]:
# Create and train models
logreg = LogisticRegression()
logreg.fit(X_train, y_train)

linreg = LinearRegression()
linreg.fit(X_train, y_train)

# Make predictions
y_pred_class = logreg.predict(X_test)
y_pred_reg = linreg.predict(X_test)

# Evaluate models
class_accuracy = accuracy_score(y_test, y_pred_class)
reg_r2 = r2_score(y_test, y_pred_reg)

print(f"Classification Accuracy: {class_accuracy}")
print(f"Regression R2 Score: {reg_r2}")

Classification Accuracy: 0.02185792349726776
Regression R2 Score: -0.005030310792471182


In [12]:
# Decision tree (Modelo) para classificacao e random forest para regressao

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestRegressor

# Create and train models
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

rf_model = RandomForestRegressor()
rf_model.fit(X_train, y_train)

# Make predictions
y_pred_class = dt_model.predict(X_test)
y_pred_reg = rf_model.predict(X_test)

# Evaluate models
class_accuracy = accuracy_score(y_test, y_pred_class)
reg_r2 = r2_score(y_test, y_pred_reg)

print(f"Classification Accuracy: {class_accuracy}")
print(f"Regression R2 Score: {reg_r2}")

Classification Accuracy: 0.0273224043715847
Regression R2 Score: 0.016601201109627595


In [13]:
# SVM para classificacao
from sklearn.svm import SVC
from sklearn.ensemble import GradientBoostingRegressor

# Create and train models
svm_model = SVC()
svm_model.fit(X_train, y_train)

gb_model = GradientBoostingRegressor()
gb_model.fit(X_train, y_train)

# Make predictions
y_pred_class = svm_model.predict(X_test)
y_pred_reg = gb_model.predict(X_test)

# Evaluate models
class_accuracy = accuracy_score(y_test, y_pred_class)
reg_r2 = r2_score(y_test, y_pred_reg)

print(f"Classification Accuracy: {class_accuracy}")
print(f"Regression R2 Score: {reg_r2}")

Classification Accuracy: 0.02185792349726776
Regression R2 Score: 0.022513615635485373


In [15]:
# Modelo de Regressao linear para predizer Consumo de massa no item em (Kg/100pcs)
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score

# Load the dataset
data = pd.read_csv("IJ-044.csv")

# Drop unnecessary columns
data = data.drop(
    [
        "Data de Produção",
        "Cód. Ordem",
        "Cód. Recurso",
        "Cód. Un.",
        "Descrição da massa (Composto)",
    ],
    axis=1,
)

In [16]:
# One-hot encode `Cód. Produto`
data = pd.get_dummies(data, columns=["Cód. Produto"])

# Split data into features (X) and target variable (y)
X = data.drop("Consumo de massa no item em (Kg/100pçs)", axis=1)
y = data["Consumo de massa no item em (Kg/100pçs)"]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

In [17]:
# Scale the data
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Create and train a Linear Regression model
linreg_model = LinearRegression()
linreg_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = linreg_model.predict(X_test)

# Evaluate the model
r2 = r2_score(y_test, y_pred)
print(f"R2 Score: {r2}")

R2 Score: 1.0


In [19]:
# Classificacao de Cod. Produto baseado nas outras variáveis:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

# Load the dataset
data = pd.read_csv("IJ-044.csv")

# Drop unnecessary columns
data = data.drop(
    [
        "Data de Produção",
        "Cód. Ordem",
        "Cód. Recurso",
        "Cód. Un.",
        "Descrição da massa (Composto)",
    ],
    axis=1,
)

# One-hot encode `Cód. Produto`
data = pd.get_dummies(data, columns=["Cód. Produto"])

In [20]:
# Split data into features (X) and target variable (y)
X = data.drop("Qtd. Produzida", axis=1)
y = data["Qtd. Produzida"]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale the data
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Create and train a Logistic Regression model
logreg_model = LogisticRegression()
logreg_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = logreg_model.predict(X_test)

# Evaluate the model
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 0.00819672131147541


In [26]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load the dataset
data = pd.read_csv("IJ-044.csv")

# Drop unnecessary columns
data = data.drop(
    [
        "Data de Produção",
        "Cód. Ordem",
        "Cód. Recurso",
        "Cód. Un.",
        "Descrição da massa (Composto)",
    ],
    axis=1,
)

# One-hot encode `Cód. Produto`
data = pd.get_dummies(data, columns=["Cód. Produto"])

# Split data into features (X) and target variable (y)
X = data.drop("Qtd. Produzida", axis=1)
y = data["Qtd. Produzida"]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale the data
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Create and train a Decision Tree model for classification
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = dt_model.predict(X_test)

# Evaluate the model's performance using accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 0.01639344262295082


In [28]:
# Predicao Cód Produto considerando o Random Forest
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Load the dataset
data = pd.read_csv("IJ-044.csv")

# Drop unnecessary columns
data = data.drop(
    [
        "Data de Produção",
        "Cód. Ordem",
        "Cód. Recurso",
        "Cód. Un.",
        "Descrição da massa (Composto)",
    ],
    axis=1,
)

# One-hot encode `Cód. Produto`
data = pd.get_dummies(data, columns=["Cód. Produto"])

# Split data into features (X) and target variable (y)
X = data.drop("Qtd. Produzida", axis=1)
y = data["Qtd. Produzida"]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale the data
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Create and train a Decision Tree model for classification
dt_model = DecisionTreeClassifier()
dt_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = dt_model.predict(X_test)

# Evaluate the model's performance using accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 0.01639344262295082


In [29]:
# Predicao Cód Produto utilizando Support Vector Machines:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import MinMaxScaler
from sklearn.svm import SVC
from sklearn.metrics import accuracy_score

# Load the dataset
data = pd.read_csv("IJ-044.csv")

# Drop unnecessary columns
data = data.drop(
    [
        "Data de Produção",
        "Cód. Ordem",
        "Cód. Recurso",
        "Cód. Un.",
        "Descrição da massa (Composto)",
    ],
    axis=1,
)

# One-hot encode `Cód. Produto`
data = pd.get_dummies(data, columns=["Cód. Produto"])

# Split data into features (X) and target variable (y)
X = data.drop("Qtd. Produzida", axis=1)
y = data["Qtd. Produzida"]

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Scale the data
scaler = MinMaxScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

# Create and train an SVM model for classification
svm_model = SVC()
svm_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = svm_model.predict(X_test)

# Evaluate the model's performance using accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Accuracy: {accuracy}")

Accuracy: 0.00819672131147541
